# 05 · 3D map of the soil parameters on the classifier grid

1. Reads the classifier's 3D grid (`settings.toml → [map] grid_csv`: 50 m columns, 0.5 m cells, P(Clay)).
2. Turns it into 1.5 m layers per column with the same inputs as the training layers (Easting, Northing, mid elevation, mean P(Clay)).
3. Predicts every target with the **final model chosen in notebook 03** (frozen in notebook 04; run 04 first).
4. Calculates **PI = LL − PL** from the predicted LL and PL (derived, not modelled).
5. Adds the **uncertainty** of each target: the CV RMSE of the final model in the layer's depth band (notebook 03).
6. Keeps the classifier's `dist_nearest_bh_m` and `reliability_zone`: beyond about 100 m from a borehole the map is close to the depth trend.

The map (it shows locations) goes to `outputs/05_map/private/` (not in git). **Run time:** a few minutes.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

O, PRIV = H.out_dir(S, "05_map"), H.out_dir(S, "05_map", private=True)
choice = pd.read_csv(H.out_dir(S, "03_results") / "model_choice.csv").set_index("target").selected
FPRIV = H.out_dir(S, "04_final_external", private=True)
models = {}
for t, name in choice.items():
    f = FPRIV / f"final_{t}_{name}.joblib"
    if not f.exists():
        raise FileNotFoundError(f"{f.name} missing: run notebook 04 first")
    models[t] = (name, joblib.load(f)["model"])
display(choice)
grid = pd.read_csv(S["map"]["grid_csv"])
print(f"Grid: {len(grid):,} cells, {grid[['X', 'Y']].drop_duplicates().shape[0]:,} columns")
lay = H.grid_layers(grid, S)
print(f"Map layers: {len(lay):,}")

## Predict

In [ ]:
dep = pd.read_csv(H.out_dir(S, "03_results") / "scores_by_depth.csv")
band = lambda d: f"{np.floor(d / 3.0) * 3:g}-{np.floor(d / 3.0) * 3 + 3:g} m"
lay["depth_band"] = lay.depth_m.map(band)
for t, (name, m) in models.items():
    pred = np.empty(len(lay))
    for i in range(0, len(lay), 200_000):
        part = lay.iloc[i:i + 200_000]
        pred[i:i + 200_000] = m.predict(H.design(part), part.depth_m.values)
    lay[t] = pred
    r = dep[(dep.target == t) & (dep.model == name)].set_index("depth_band").rmse
    lay[f"{t}_rmse"] = lay.depth_band.map(r).values
    print(f"{t:<16} model {name:<6} mean {pred.mean():.2f}  range {pred.min():.2f} - {pred.max():.2f}")
if {"LL", "PL"} <= set(models):
    lay["PI"] = (lay.LL - lay.PL).clip(lower=0)
    print("PI = LL - PL: mean", round(lay.PI.mean(), 2), "| layers with LL < PL set to 0:", int((lay.LL < lay.PL).sum()))
lay.to_csv(PRIV / "soil_parameters_3d_layers.csv", index=False)
print("Saved", PRIV / "soil_parameters_3d_layers.csv")

## Summary by depth (no locations)

In [ ]:
cols = [c for c in [*models, "PI"] if c in lay]
by_depth = lay.groupby("depth_m")[cols].mean()
by_zone = lay.groupby("reliability_zone")[cols].mean()
by_depth.to_csv(O / "mean_by_depth.csv"); by_zone.to_csv(O / "mean_by_reliability_zone.csv")
display(by_depth.round(2)); display(by_zone.round(2))
fig, axes = plt.subplots(1, len(cols), figsize=(2.8 * len(cols), 4.5), sharey=True)
for ax, c in zip(np.atleast_1d(axes), cols):
    g = lay.groupby("depth_m")[c]
    ax.fill_betweenx(g.mean().index, g.quantile(0.1), g.quantile(0.9), alpha=0.3)
    ax.plot(g.mean(), g.mean().index, marker="o")
    ax.set_title(c, fontsize=9)
np.atleast_1d(axes)[0].invert_yaxis(); np.atleast_1d(axes)[0].set_ylabel("depth (m)")
plt.tight_layout(); plt.savefig(O / "map_profiles_by_depth.png", dpi=200); plt.show()
print(f"Finished in {(time.time() - T0) / 60:.1f} min.")